In [ ]:
import aiogram
import asyncio
import logging
import sys
from datetime import datetime, timedelta

from aiogram import Bot, Dispatcher, Router, F
from aiogram.enums import ParseMode
from aiogram.types import InlineKeyboardButton, InlineKeyboardMarkup, Message, CallbackQuery, FSInputFile
from aiogram.filters import CommandStart, Command, StateFilter
from aiogram.client.default import DefaultBotProperties
from aiogram.fsm.context import FSMContext
from aiogram.fsm.state import State, StatesGroup
from aiogram.fsm.storage.memory import MemoryStorage

from PIL import Image, ImageDraw, ImageFont
from datetime import datetime, timedelta, date
import calendar

TOKEN = 'x'
bot = Bot(token=TOKEN, default=DefaultBotProperties(parse_mode=ParseMode.HTML))

storage = MemoryStorage()
dp = Dispatcher(storage=storage)
router = Router()

dp.include_router(router)

## Todo list

tasks_dict = {}
calendar_dict = {}

class TodoStates(StatesGroup):
    waiting_for_tasks = State()
    waiting_for_date = State()
    waiting_for_date_done = State()


class CalendarStates(StatesGroup):
    waiting_for_dateC = State()
    waiting_for_tasksC = State()
    waiting_for_date_timeC = State()
    waiting_for_date_shareC = State()
    waiting_for_username = State()




def find_number(task):
    num = ''
    i = 0

    while task[i] != '.':
        num += task[i]
        i += 1
    return int(num)


def tasks_done(done_tasks_list: list[str], all_tasks_list: list[str]):
    new_tasks = []

    for task in all_tasks_list:
        new_task = [i.strip() for i in task.split('.')]
        if not task:
            continue
        if new_task[0] in done_tasks_list or new_task[1] in done_tasks_list:
            new_tasks.append(f"<s>{''.join(task)}</s>")
        else:
            new_tasks.append(task)
    
    return new_tasks
        

def validate_and_parse_date(text):
    today = datetime.now().date()
    date_str = today.strftime('%d.%m')
    
    if text == 'сегодня':
        return date_str
    elif text == 'завтра':
        return (today + timedelta(days=1)).strftime('%d.%m')
    else:
        try:
            dt = datetime.strptime(text.strip(), '%d.%m')
            return dt.strftime('%d.%m')
        except ValueError:
            return None
        

def save_info(dict_for_save):
    if dict_for_save:
        with open('storage.txt', 'w') as f:
            for key, el in dict_for_save.items():
                f.write(key + '\n')
                for task in el:
                    f.write(task + '\n')
        f.close()


def open_f(dict_for_take):
    with open('storage.txt', 'r') as f:
        file = [i for i in f.readlines()]
        if file:
            for line in file:
                mb_date = validate_and_parse_date(line[:-1])
                if mb_date != None:
                    key = mb_date
                else:
                    if key not in dict_for_take:
                        dict_for_take[key] = [line[:-1]]
                    else:
                        dict_for_take[key].append(line[:-1])
    f.close()
open_f(tasks_dict)
    

## Команда /start: описание бота, действия с календарем и todo листом
@dp.message(CommandStart())
async def command_start_handler(message: Message) -> None:
    keyboard = InlineKeyboardMarkup(
        inline_keyboard=[
            [
                InlineKeyboardButton(text='todo list', callback_data='cmd_todo_list'),
                InlineKeyboardButton(text='calendar', callback_data='cmd_calendar')
            ]
        ]
    )

    sent_message = await message.answer(f"здравствуйте, {aiogram.html.bold(message.from_user.full_name)}!\n\n"
                         "в этом боте вы сможете:\n\n1) создать список дел, добавить в него любые дела в любой момент;\n"
                         "отметить дело или дела выполнеными;\nпосмотреть дела на сегодня\n\n"
                         "2) посмотреть календарь в различных форматах;\nзанять слот в календаре делом;\nподелится календарем с другим человеком, есть возможность скрыть все дела или оставить их открытыми\n\n"
                         "что откроем?", reply_markup=keyboard)
    await message.chat.pin_message(sent_message.message_id, disable_notification=True)


## todo list: получение списка дел, их обработка; получение даты и ее обработка; сохранение данных
@dp.callback_query(F.data == 'cmd_todo_list')
async def firsttodo_list_handler(callback_query: CallbackQuery):
    await todo_buttons(callback_query, "выберите действие с todo листом:"
                       "возможные действия:\n\n1) создать список дел или добавить дела в существующий список -> нажмите на кнопку 📋\n"
                        "2) отметить дело выполненым -> нажмите на кнопку ✅\n"
                        "3) посмотреть todo на сегодня -> нажмите на кнопку 👀")


@dp.callback_query(lambda c: c.data.startswith('todo_'))
async def todo_list_handler(callback_query: CallbackQuery, state: FSMContext):
    todo_command = callback_query.data[5:]
    await state.update_data(todo_command=todo_command)

    if todo_command == 'create_tasks':
        await callback_query.message.answer("на какой день нужно записать дела?\n"
                                        "напишите дату в формате ДД.ММ или текстом, если это сегодня или завтра\n"
                                        "например:\n"
                                        "'сегодня' или '25.01'" )
        
        await state.set_state(TodoStates.waiting_for_date)
    elif todo_command == 'done_tasks':
        await callback_query.message.answer("у какого дня надо отметить выполненым дело или дела\n"
                                        "напишите дату в формате <b>ДД.ММ</b> или текстом, если это сегодня или завтра\n\n"
                                        "например:\n"
                                        "'сегодня' или '25.01'" )
        await state.set_state(TodoStates.waiting_for_date)
    elif todo_command == 'see_tasks':
        date = datetime.now().date()
        date_str = date.strftime('%d.%m')
        if date_str in tasks_dict:
            new_list = [''.join(tasks_dict[date_str][i]) for i in range(len(tasks_dict[date_str]))]
            tasks_useble_text = '\n'.join(new_list)
            await callback_query.message.answer(f"список дел на сегодня:\n\n{tasks_useble_text}", parse_mode='HTML')
            await todo_buttons(callback_query, "что дальше")
        else:
            await callback_query.message.answer("похоже у вас нет дел на сегодня. составим список?")

    await callback_query.answer()



@dp.message(TodoStates.waiting_for_date)
async def process_date(message: Message, state: FSMContext):
    user_date = message.text.strip().lower()
    normalized_date = validate_and_parse_date(user_date)

    if normalized_date is None:
        await message.answer("<b>неверный формат даты</b>\n\nнапишите дату в формате <b>ДД.ММ</b> или сеогдня, завтра", parse_mode='HTML')
        return
    await state.update_data(date=normalized_date)

    data = await state.get_data()
    todo_command = data.get('todo_command')

    if todo_command == 'create_tasks':
        await message.answer(f"отлично! записываем дела на <b>{normalized_date}</b>\n\n"
                         "теперь напишите список дел на этот день.\n"
                         "список дел напишите через запятую")
        await state.set_state(TodoStates.waiting_for_tasks)
    elif todo_command == 'done_tasks':
        if normalized_date in tasks_dict:
            await message.answer(f"отлично! отмечаем выполнеными дела на <b>{normalized_date}</b>\n\n")
            await message.answer(f"вот список дел на {normalized_date}:\n\n{'\n'.join(tasks_dict[normalized_date])}\n\nтеперь напишите список дел или номера дел, которые надо отметить выполнеными.\nсписок дел/номера дел напишите через запятую", parse_mode='HTML')
            await state.set_state(TodoStates.waiting_for_tasks)
        else:
            await message.answer(f"на <b>{normalized_date}</b> пока нет дел, возможно вы ввели не ту дату\n\n")
            keyboard = InlineKeyboardMarkup(
                inline_keyboard=[ 
                    [
                        InlineKeyboardButton(text='создать список на эту дату', callback_data='done_creat'),
                        InlineKeyboardButton(text='изменить дату', callback_data='done_change'),
                    ]
                ]
            )
            await message.answer("что сделаем", reply_markup=keyboard)
 


@dp.callback_query(lambda c: c.data.startswith('done_'))
async def handle_done_buttons(callback_query: CallbackQuery, state: FSMContext):
    command = callback_query.data

    if command == 'done_creat':
        await state.update_data(todo_command='create_tasks')

        data = await state.get_data()
        date = data.get('date', 'Неизвестная дата')

        await callback_query.message.edit_text(f"отлично! записываем дела на <b>{date}</b>\n\n"
                         "теперь напишите список дел на этот день.\n"
                         "список дел напишите через запятую")
        
        await state.set_state(TodoStates.waiting_for_tasks)
    elif command == 'done_change':
        await callback_query.message.answer("введите другую дату")
        await state.set_state(TodoStates.waiting_for_date)
    
    await callback_query.answer()


@dp.message(TodoStates.waiting_for_tasks)
async def process_tasks(message: Message, state: FSMContext):
    data = await state.get_data()
    date = data.get('date', 'Неизвестная дата').strip().lower()

    tasks_text = message.text.strip()
    tasks_text_list = [i.strip().lower() for i in tasks_text.split(',')]

    todo_command = data.get('todo_command')

    if date in tasks_dict:
        if todo_command == 'create_tasks':
            last_number = find_number(tasks_dict[date][-1]) + 1
            for i in range(len(tasks_text_list)):
                tasks_text_list[i] = str(last_number) + '. ' + tasks_text_list[i]
                last_number += 1
            tasks_dict[date] += tasks_text_list

        elif todo_command == 'done_tasks':
            tasks_dict[date] = tasks_done(tasks_text_list, tasks_dict[date])
    elif date not in tasks_dict and todo_command == 'create_tasks':
        tasks_text_list = [str(i + 1) + '. ' + tasks_text_list[i] for i in range(len(tasks_text_list))]
        tasks_dict[date] = tasks_text_list

    tasks_useble_text = '\n'.join(tasks_dict[date])
    save_info(tasks_dict)

    await message.answer(f"список дел на {date}:\n\n{tasks_useble_text}", parse_mode='HTML')

    await state.clear()
    await todo_buttons(message, "что дальше?\n"
                       "возможные действия:\n\n1) создать список дел или добавить дела в существующий список -> нажмите на кнопку 📋\n"
                         "2) отметить дело выполненым -> нажмите на кнопку ✅\n"
                         "3) посмотреть todo на сегодня -> нажмите на кнопку 👀")



async def todo_buttons(message_or_query, text: str = "что делаем?"):
    keyboard = InlineKeyboardMarkup(
        inline_keyboard=[
            [
                InlineKeyboardButton(text='📋', callback_data='todo_create_tasks'),
                InlineKeyboardButton(text='✅', callback_data='todo_done_tasks'),
                InlineKeyboardButton(text='👀', callback_data='todo_see_tasks')
            ]
        ]
    )
    if hasattr(message_or_query, 'message'):
        await message_or_query.message.answer(text, reply_markup=keyboard)
        await message_or_query.answer()
    else:
        await message_or_query.answer(text, reply_markup=keyboard)



## календарь

def day_of_the_week(user_date):
    days = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']

    day, month = [int(i) for i in user_date.split('.')]
    right_date = date(2026, month, day)
    
    day_week = days[right_date.weekday()]
    return day_week


def normalize_time(str_time):
    time = f"{str_time.zfill(2)}:00"
    return time

def dot_line(image, x, y):
    for i in range(x, 990, 8):
        image.circle(xy=(i, y), radius=1.5, fill='black')

def find_time(text):
    time = ''
    i = 0
    while text[i].isdigit():
        time += text[i]
        i += 1
    return (time, i)


# 1 день из календаря
def show_calendar_day(user_date, all_info):
    font_regular = ImageFont.truetype("Anonymous.ttf", 34)
    font_bold = ImageFont.truetype("Anonymous_Bold.ttf", 40)

    width, height = 1080, 1600
    img = Image.new('RGB', (width, height), (253, 245, 230))
    draw = ImageDraw.Draw(img)

    draw.text(text='дата' + ' ' + user_date, xy=(30, 50), font=font_bold, fill='black')
    draw.rounded_rectangle((10, 45, 290, 100), 15, outline='black', width=3)
    draw.text(text=day_of_the_week(user_date), xy=(350, 50), font=font_bold, fill='black')

    cor = []

    x = 10
    y = 150

    for i in range(24):
        time = normalize_time(str(i))
        draw.text(text=time, xy=(x, y), font=font_regular, fill='black')
        x += 100
        cor.append((x + 30, y - 10))
        dot_line(draw, x, y + 30)
        x = 10
        y += 60
    
    if user_date in all_info:
        for el in all_info[user_date]:
            time, index = find_time(el)
            task = el[index:]
            x, y = cor[int(time)]
            draw.text(text=task, xy=(x, y), font=font_regular, fill='black')

    # img.show()
    name = 'calendar.png'
    img.save(name)
    return name


# месяц календаря

days = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']


def day_of_the_week(user_date):
    day, month = [int(i) for i in user_date.split('.')]
    right_date = date(2026, month, day)
    
    day_week = days[right_date.weekday()]
    return day_week


def find_time(text):
    time = ''
    i = 0

    while text[i].isdigit():
        time += text[i]
        i += 1
    return (time, i)


def normalize_time(str_time):
    time = f"{str_time.zfill(2)}:00"
    return time


def not_enough_space(text, dif, time, mn):
    if (len(text) + 6) * mn >= dif - 20:
        new_text = []
        new_line = ''
        text_list = text.split()
        text_list.insert(0, time)
        for el in text_list:
            if len(new_line + el + ' ') * mn < dif - 20:
                new_line += el + ' '
            else:
                new_text.append(new_line)
                new_line = el
        if new_line:
            new_text.append(new_line)
        return new_text
    return None



def show_calendar_month(month_num, all_info):
    font_sizeB, font_sizeR = 50, 45

    font_regular = ImageFont.truetype("Anonymous.ttf", font_sizeR)
    font_bold = ImageFont.truetype("Anonymous_Bold.ttf", font_sizeB)

    width, height = 3602, 2480
    img = Image.new('RGB', (width, height), (253, 245, 230))
    draw = ImageDraw.Draw(img)

    cor_x = []

    dif_height = (height - 100) // 5
    dif_width = (width - 50) // 7
    x, y = dif_width // 2 - font_sizeB // 2, 30

    j = 0
    for i in range(0, width - dif_width, dif_width):
        draw.text(text=days[j], xy=(i + x, y), font=font_bold, fill='black')
        j += 1

    y = 100
    cor_x.append(25)
    for i in range(dif_width, width - dif_width, dif_width):
        draw.line([(i, y), (i, height - 1)], fill='black', width=0)
        cor_x.append(i)

    dict_cor_x = dict(zip(days, cor_x))


    error_rate = 30
    height_limit = dif_height - (2 * font_sizeR + error_rate)
    month = int(month_num)
    last_day = calendar.monthrange(2026, month)[1]
    for day in range(1, last_day + 1):
        count_lines = 1
        tr = True

        normal_date = f"{day:02d}.{month:02d}"
        week_d = day_of_the_week(normal_date)
        x_cor = dict_cor_x[day_of_the_week(normal_date)] + error_rate

        draw.text(text=normal_date, xy=(x_cor + error_rate, y), font=font_regular, fill='black')
        draw.rounded_rectangle((x_cor - 10, y - 10, font_sizeR * 4 + x_cor, font_sizeR + y + 10), 15, outline='black', width=3)

        tasks_y = y + font_sizeR + error_rate
        if normal_date in all_info:
            for info in all_info[normal_date]:
                tm, index = find_time(info)
                
                task = info[index:]
                time = normalize_time(tm)

                parsed_task = not_enough_space(task, dif_width, time, font_sizeR // 2)
                if parsed_task:
                    for i in range(len(parsed_task)):
                        draw.text(text=parsed_task[i], xy=(x_cor, tasks_y), font=font_regular, fill='black')
                        tasks_y += font_sizeR
                        count_lines += 1
                        if count_lines * font_sizeR >= height_limit:
                            tr = False
                            break
                else:
                    draw.text(text=(time + task), xy=(x_cor, tasks_y), font=font_regular, fill='black')
                    tasks_y += font_sizeR
                    count_lines += 1
                if count_lines * font_sizeR >= height_limit or not(tr):
                    draw.text(text='...', xy=(x_cor, tasks_y), font=font_regular, fill='black')
                    break
        if week_d == 'Вс':
            y += dif_height 

    name = 'calendar.png'
    img.save(name)
    return name




def correct_tasks(info):
    new_info = ' '.join(info)
    info_list = []
    if ';' in new_info:
        info_list = new_info.split(';')
    else:
        info_list.append(new_info)
    return info_list


def select_time(items):
    time = ''
    i = 0
    while items[i].isdigit():
        time += items[i]
        i += 1
    return int(time)





## выбор действий с календарем: открыть календарь; закрыть слот; поделится календарем
@dp.callback_query(F.data == 'cmd_calendar')
async def calendar_handler(callback_query: CallbackQuery):
    await calendar_buttons(callback_query, "выберите действие с календарем"
                       "возможные действия:\n\n1) открыть календарь -> нажмите на кнопку 📆\n"
                        "2) закрыть слот в календаре -> нажмите на кнопку 📌\n"
                        "3) поделится календарем -> нажмите на кнопку 🔗")


###
@dp.callback_query(lambda c: c.data.startswith('cmd_'))
async def open_calendar(callback_query: CallbackQuery, state: FSMContext):
    command = callback_query.data[4:]
    if command == 'open_calendar':
        keyboard = InlineKeyboardMarkup(
            inline_keyboard=[
                [
                    InlineKeyboardButton(text='день', callback_data='open_day'),
                    InlineKeyboardButton(text='месяц', callback_data='open_month'),
                    InlineKeyboardButton(text='другое', callback_data='open_other')
                ]
            ]
        )

        await callback_query.message.edit_text("в каком формате открыть календарь?", reply_markup=keyboard)
    elif command == 'close_slot':
        await callback_query.message.answer("введите дату, когда нужно закрыть слот\n\n")
        await state.set_state(CalendarStates.waiting_for_date_timeC)

    elif command == 'share_calendar':
        command = callback_query.data[4:]
        keyboard = InlineKeyboardMarkup(
            inline_keyboard=[
                [
                    InlineKeyboardButton(text='да', callback_data='user_yes'),
                    InlineKeyboardButton(text='нет', callback_data='user_no')
                ]
            ]
        )

        await callback_query.message.edit_text("человек, которому вы хотите отправить календарь использует этот бот?", reply_markup=keyboard)
    
    await callback_query.answer()


@dp.callback_query(lambda c: c.data.startswith('open_'))
async def open_calendar(callback_query: CallbackQuery, state: FSMContext):
    command = callback_query.data[5:]
    await state.update_data(open_or_share='open')
    await state.update_data(format_to_open=command)

    if command == 'day':
        await callback_query.message.edit_text("введите день, который нужно показать\n\nнапишите дату в формате ДД.ММ или текстом, если это сегодня или завтра\n"
                                        "например:\n"
                                        "'сегодня' или '25.01'")
        await state.set_state(CalendarStates.waiting_for_dateC)
    elif command == 'month':
        await callback_query.message.edit_text("введите номер или название месяца, который нужно показать\n\n"
                                        "например:\n"
                                        "'февраль' или '2'")
        await state.set_state(CalendarStates.waiting_for_dateC)
    elif command == 'other':
        await callback_query.message.edit_text("введите отрезок, который нужно показать\n\nотрезок напишите в формате ДД.ММ:ДД.ММ или от ДД.ММ до ДД.ММ"
                                        "например:\n"
                                        "'12.01:25.01' или 'от 3.10 до 10.11'")
        await state.set_state(CalendarStates.waiting_for_dateC)
        

@dp.message(CalendarStates.waiting_for_dateC)
async def process_date(message: Message, state: FSMContext):
    user_date = message.text.strip().lower()
    data = await state.get_data()
    open_or_share = data.get('open_or_share')

    if open_or_share == 'open':
        format_to_open = data.get('format_to_open')

        if format_to_open == 'day':
            normalized_date = validate_and_parse_date(user_date)
            if normalized_date is None:
                await message.answer("<b>неверный формат даты</b>\n\nнапишите дату в формате <b>ДД.ММ</b> или сеогдня, завтра", parse_mode='HTML')
                return
            await state.update_data(dateC=normalized_date)

            # await state.clear()

            photo_name = show_calendar_day(normalized_date, calendar_dict)
            photo_path = FSInputFile(photo_name)
            
            await bot.send_photo(chat_id=message.chat.id, photo=photo_path, caption=f"календарь на {normalized_date}")
        
        elif format_to_open == 'month':
            month_name = ['январь', 'февраль', 'март', 'апрель', 'май', 'июнь', 'июль', 'август', 'сентябрь', 'октябрь', 'ноябрь', 'декабрь']
            
            try:
                num = int(user_date)
                if not(num >= 1 and num <= 12):
                    await message.answer("<b>месяц должен быть от 1 до 12</b>\nпопробуйте снова", parse_mode='HTML')
                    return
            
                photo_name = show_calendar_month(user_date, calendar_dict)
                photo_path = FSInputFile(photo_name)

                await bot.send_photo(chat_id=message.chat.id, photo=photo_path, caption=f"Календарь на {month_name[num - 1]}")
            except ValueError:
                name = user_date.strip().lower()
                if name not in month_name:
                    await message.answer("<b>неверный формат.</b>\nвведите число от 1 до 12 или названия месяца. например: февраль")
                    return
                
                photo_name = show_calendar_month(month_name.index(name) + 1, calendar_dict)
                photo_path = FSInputFile(photo_name)

                await bot.send_photo(chat_id=message.chat.id, photo=photo_path, caption=f"календарь на {user_date}")
        elif format_to_open == 'other':
            pass
    
    elif open_or_share == 'share':
        format_to_share = data.get_data('format_to_share')
        user_name = data.get_data('give_username')

        if format_to_share == 'day':
            normalized_date = validate_and_parse_date(user_date)
            if normalized_date is None:
                await message.answer("<b>неверный формат даты</b>\n\nнапишите дату в формате <b>ДД.ММ</b> или сеогдня, завтра", parse_mode='HTML')
                return
            await state.update_data(dateC=normalized_date)
        
        elif format_to_share == 'month':
            month_name = ['январь', 'февраль', 'март', 'апрель', 'май', 'июнь', 'июль', 'август', 'сентябрь', 'октябрь', 'ноябрь', 'декабрь']
            
            try:
                num = int(user_date)
                if not(num >= 1 and num <= 12):
                    await message.answer("<b>месяц должен быть от 1 до 12</b>\nпопробуйте снова", parse_mode='HTML')
                    return
            
                # photo_name = show_calendar_month(user_date, calendar_dict)
                # photo_path = FSInputFile(photo_name)

                # await bot.send_photo(chat_id=message.chat.id, photo=photo_path, caption=f"Календарь на {month_name[num - 1]}")
            except ValueError:
                name = user_date.strip().lower()
                if name not in month_name:
                    await message.answer("<b>неверный формат.</b>\nвведите число от 1 до 12 или названия месяца. например: февраль")
                    return
                
                # photo_name = show_calendar_month(month_name.index(name) + 1, calendar_dict)
                # photo_path = FSInputFile(photo_name)

                # await bot.send_photo(chat_id=message.chat.id, photo=photo_path, caption=f"календарь на {user_date}")
        elif format_to_share == 'other':
            pass
            
    
    await state.clear()


    await calendar_buttons(message, "что делаем дальше? "
                       "возможные действия:\n\n1) открыть календарь -> нажмите на кнопку 📆\n"
                        "2) закрыть слот в календаре -> нажмите на кнопку 📌\n"
                        "3) поделится календарем -> нажмите на кнопку 🔗")

        

    
@dp.message(CalendarStates.waiting_for_date_timeC)
async def close_slot(message: Message, state: FSMContext):
    user_date = message.text.strip().lower()
    normalized_date = validate_and_parse_date(user_date)

    if normalized_date is None:
        await message.answer("<b>неверный формат даты</b>\n\nнапишите дату в формате <b>ДД.ММ</b> или сеогдня, завтра", parse_mode='HTML')
        return
    await state.update_data(dateC=normalized_date)

    await message.answer(f"отлично! записываем дела на <b>{normalized_date}</b>\n\n"
                         "теперь напишите время и дело на этот день, если дел несколько, разделяйте их ';'\n\n<I>**время пишите одной цифрой**</I>"
                         "например:\n"
                         "'11 сходить в магазин' или '10 рабочая встреча; 17 заехать в мастерскую'")
    await state.set_state(CalendarStates.waiting_for_tasksC)
    


@dp.message(CalendarStates.waiting_for_tasksC)
async def process_tasks(message: Message, state: FSMContext):
    global calendar_dict

    data = await state.get_data()
    date = data.get('dateC', 'Неизвестная дата').strip().lower()

    tasks_text = message.text.strip()
    tasks_list = [el.strip() for el in tasks_text.split(';')]

    if date in calendar_dict:
        calendar_dict[date].append(tasks_list)
        calendar_dict[date] = sorted(calendar_dict[date], key=select_time)
    else:
        calendar_dict[date] = tasks_list
    
    # calendar_dict = sorted(calendar_dict.items())

    await state.clear()
    await calendar_buttons(message, "что делаем дальше? "
                       "возможные действия:\n\n1) открыть календарь -> нажмите на кнопку 📆\n"
                        "2) закрыть слот в календаре -> нажмите на кнопку ✍🏻\n"
                        "3) поделится календарем -> нажмите на кнопку 🔗")



@dp.callback_query(lambda c: c.data.startswith('user_'))
async def share_calendar(callback_query: CallbackQuery, state: FSMContext):
    command = callback_query.data[5:]
    
    if command == 'yes':
        await callback_query.message.edit_text("введите user name человека, которому хотите отправить календарь?")
        await state.set_state(CalendarStates.waiting_for_username)
    else:
        await state.update_data(give_usernamr=None)
    
    keyboard = InlineKeyboardMarkup(
            inline_keyboard=[
                [
                    InlineKeyboardButton(text='да', callback_data='Sopen_yes'),
                    InlineKeyboardButton(text='нет', callback_data='Sopen_no'),
                    InlineKeyboardButton(text='не полностью', callback_data='Sopen_other'),
                ]
            ]
        )

    await callback_query.message.edit_text("нужно ли скрыть все дела?", reply_markup=keyboard)
    


@dp.callback_query(lambda c: c.data.startswith('Sopen_'))
async def share_calendar(callback_query: CallbackQuery, state: FSMContext):
    command = callback_query.data[6:]
    await state.update_data(share_open=command)

    if command == 'yes' or command == 'no':
        keyboard = InlineKeyboardMarkup(
            inline_keyboard=[
                [
                    InlineKeyboardButton(text='день', callback_data='share_day'),
                    InlineKeyboardButton(text='месяц', callback_data='share_month'),
                    InlineKeyboardButton(text='другое', callback_data='share_other'),
                ]
            ]
        )

        await callback_query.message.edit_text("в каком формате нужно отправить календарь?", reply_markup=keyboard)
    else:
        keyboard = InlineKeyboardMarkup(
            inline_keyboard=[
                [
                    InlineKeyboardButton(text='да', callback_data='a_lot_of_questions'),
                    InlineKeyboardButton(text='нет', callback_data='user_no')
                ]
            ]
        )

        await callback_query.message.edit_text("сейчас будет N вопросов для корекктной работы бота, хотите ли вы на них отвечать?", reply_markup=keyboard)
    
    

@dp.callback_query(lambda c: c.data.startswith('share_'))
async def share_calendar(callback_query: CallbackQuery, state: FSMContext):
    command = callback_query.data[6:]
    await state.update_data(format_to_share=command)
    await state.update_data(open_or_share='share')
    # await state.update_data(share_open=command)

    if command == 'day':
        await callback_query.message.edit_text("введите день, который нужно отправить\n\nнапишите дату в формате ДД.ММ или текстом, если это сегодня или завтра\n"
                                        "например:\n"
                                        "'сегодня' или '25.01'")
    elif command == 'month':
        await callback_query.message.edit_text("введите номер или название месяца, который нужно отправить\n\n"
                                        "например:\n"
                                        "'февраль' или '2'")
    elif command == 'other':
        await callback_query.message.edit_text("введите отрезок, который нужно отправить\n\nотрезок напишите в формате ДД.ММ:ДД.ММ или от ДД.ММ до ДД.ММ"
                                        "например:\n"
                                        "'12.01:25.01' или 'от 3.10 до 10.11'")
    await state.set_state(CalendarStates.waiting_for_dateC)




async def calendar_buttons(message_or_query, text: str = "что делаем?"):
    keyboard = InlineKeyboardMarkup(
        inline_keyboard=[
            [
                InlineKeyboardButton(text='📆', callback_data='cmd_open_calendar'),
                InlineKeyboardButton(text='📌', callback_data='cmd_close_slot'),
                InlineKeyboardButton(text='🔗', callback_data='cmd_share_calendar')
            ]
        ]
    )

    if hasattr(message_or_query, 'message'):
        await message_or_query.message.answer(text, reply_markup=keyboard)
        await message_or_query.answer()
    else:
        await message_or_query.answer(text, reply_markup=keyboard)





@dp.message()
async def debug_any_message(message: Message):
    print("Any message")


@dp.callback_query()
async def debug_any_callback(callback_query: CallbackQuery):
    await callback_query.answer()


async def main() -> None:
    await dp.start_polling(bot)

import nest_asyncio
nest_asyncio.apply()

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

await main()

{'02.01': ['1. купить молоко', '2. убраться', '3. почитать', '4. позаниматься английским', '5. поймать кота'], '03.01': ['<s>1. купить воды</s>', '2. убрать квартиру', '<s>3. приготовить</s>', '4. выбрать подарки', '5. найти фильм', '6. выучить слова', '7. забрать кота', '8. забрать кота']}
INFO:aiogram.dispatcher:Start polling
INFO:aiogram.dispatcher:Run polling for bot @useble_calendar_bot id=8436152259 - 'Календарь'
INFO:aiogram.event:Update id=464713103 is handled. Duration 254 ms by bot id=8436152259
Any message
INFO:aiogram.event:Update id=464713104 is handled. Duration 1 ms by bot id=8436152259
INFO:aiogram.event:Update id=464713105 is handled. Duration 135 ms by bot id=8436152259
INFO:aiogram.event:Update id=464713106 is handled. Duration 557 ms by bot id=8436152259
INFO:aiogram.event:Update id=464713107 is handled. Duration 73 ms by bot id=8436152259
INFO:aiogram.event:Update id=464713108 is handled. Duration 76 ms by bot id=8436152259
INFO:aiogram.event:Update id=464713109 is